In [1]:
import numpy as np
from scipy.stats import norm

# ==========================================
# 1. HÀM TOÁN HỌC (CHẠY ĐỘC LẬP KHÔNG CẦN IMPORT)
# ==========================================
def diebold_mariano_test(y_true, y_pred1, y_pred2, h=1):
    e1, e2 = y_true - y_pred1, y_true - y_pred2
    d = e1**2 - e2**2
    T = len(d)
    mean_d = np.mean(d)
    
    lag = h - 1
    gamma = np.zeros(lag + 1)
    for i in range(lag + 1):
        gamma[i] = np.sum((d - mean_d)**2)/T if i == 0 else np.sum((d[i:] - mean_d) * (d[:-i] - mean_d))/T
            
    var_d = gamma[0] + sum(2 * (1.0 - i/(lag + 1)) * gamma[i] for i in range(1, lag + 1))
    if var_d == 0: return 0.0, 1.0
    
    dm_stat = mean_d / np.sqrt(var_d / T)
    p_value = 2 * (1 - norm.cdf(abs(dm_stat)))
    return float(dm_stat), float(p_value)

def clark_west_test(y_true, y_pred1, y_pred2, h=1):
    e1, e2 = y_true - y_pred1, y_true - y_pred2
    f = e1**2 - e2**2 + (y_pred1 - y_pred2)**2
    T = len(f)
    mean_f = np.mean(f)
    
    lag = h - 1
    gamma = np.zeros(lag + 1)
    for i in range(lag + 1):
        gamma[i] = np.sum((f - mean_f)**2)/T if i == 0 else np.sum((f[i:] - mean_f) * (f[:-i] - mean_f))/T
            
    var_f = gamma[0] + sum(2 * (1.0 - i/(lag + 1)) * gamma[i] for i in range(1, lag + 1))
    if var_f == 0: return 0.0, 1.0
    
    cw_stat = mean_f / np.sqrt(var_f / T)
    p_value = 1 - norm.cdf(cw_stat)
    return float(cw_stat), float(p_value)


# ==========================================
# 2. ĐỌC FILE NPZ VÀ TÍNH P-VALUE THESIS
# ==========================================
try:
    # TODO: BẠN HÃY SỬA LẠI ĐƯỜNG DẪN DƯỚI ĐÂY CHO ĐÚNG VỚI DATASET KAGGLE CỦA BẠN
    # Thường là "/kaggle/input/ten-dataset-cua-ban/baseline_predictions.npz"
    path_baseline = "/kaggle/input/datasets/namkhanhng/siginificant-test/baseline_predictions.npz"
    path_mfn = "/kaggle/input/datasets/namkhanhng/siginificant-test/mfn_test_predictions_tabular_only.npz"
    
    baseline_preds = np.load(path_baseline)
    mfn_preds = np.load(path_mfn) 
    
    print("=" * 60)
    print("STATISTICAL SIGNIFICANCE TESTING (p-values for Table 4.2)")
    print("=" * 60)
    
    targets = ["y_baseline", "y_heuristic", "y_vol_adj_return"]
    for tgt in targets:
        h = 8 if tgt == "y_baseline" else 1
        
        y_true = baseline_preds[f"{tgt}_y_true"]
        y_pred_xgb = baseline_preds[f"{tgt}_XGBoost"]
        y_pred_hist = baseline_preds[f"{tgt}_HistoricalMean"]
        
        # Sửa chữ "MFN" dưới đây cho đúng với key lưu trong file npz. 
        # Nếu model là full, key thường là f"{tgt}_MFN"
        y_pred_mfn = mfn_preds[f"{tgt}_MFN"]
        
        print(f"\nTarget: {tgt} (Horizon h={h})")
        print("-" * 40)
        
        cw_stat_mfn, cw_pval_mfn = clark_west_test(y_true, y_pred_hist, y_pred_mfn, h=h)
        sig_cw_mfn = "SIGNIFICANT" if cw_pval_mfn < 0.05 else "Not significant"
        print(f"  HistMean vs MFN:      p-value = {cw_pval_mfn:.4f} ({sig_cw_mfn})")
        
        dm_stat, dm_pval = diebold_mariano_test(y_true, y_pred_xgb, y_pred_mfn, h=h)
        sig_dm = "SIGNIFICANT" if dm_pval < 0.05 else "Not significant"
        print(f"  XGBoost vs MFN:       p-value = {dm_pval:.4f} ({sig_dm})")
        
except Exception as e:
    print("Lỗi đọc file: ", e)
    print("Bạn vui lòng kiểm tra lại chính xác đường dẫn /kaggle/input/... tới 2 file .npz nhé!")


STATISTICAL SIGNIFICANCE TESTING (p-values for Table 4.2)

Target: y_baseline (Horizon h=8)
----------------------------------------
  HistMean vs MFN:      p-value = 0.0000 (SIGNIFICANT)
  XGBoost vs MFN:       p-value = 0.0001 (SIGNIFICANT)

Target: y_heuristic (Horizon h=1)
----------------------------------------
  HistMean vs MFN:      p-value = 0.0000 (SIGNIFICANT)
  XGBoost vs MFN:       p-value = 0.7544 (Not significant)

Target: y_vol_adj_return (Horizon h=1)
----------------------------------------
  HistMean vs MFN:      p-value = 0.8402 (Not significant)
  XGBoost vs MFN:       p-value = 0.0000 (SIGNIFICANT)


In [ ]:
# ==========================================
# 3. DM TEST GIỮA CÁC ABLATION VARIANTS
# ==========================================
# So sánh trực tiếp các cấu hình MFN với nhau
# Tất cả đều dùng DM test (non-nested models, không dùng Clark-West)

import os

BASE_DIR = "/kaggle/input/datasets/namkhanhng/siginificant-test"  # <<< SỬA NẾU CẦN

# Load y_true từ baseline file (chung cho mọi so sánh)
baseline_preds = np.load(f"{BASE_DIR}/baseline_predictions.npz")

# Load predictions của 4 ablation variants
ablation_files = {
    "tabular_only" : f"{BASE_DIR}/mfn_test_predictions_tabular_only.npz",
    "no_text"      : f"{BASE_DIR}/mfn_test_predictions_no_text.npz",
    "no_image"     : f"{BASE_DIR}/mfn_test_predictions_no_image.npz",
    "full"         : f"{BASE_DIR}/mfn_test_predictions_full.npz",
}

ablation_preds = {}
for name, path in ablation_files.items():
    if os.path.exists(path):
        ablation_preds[name] = np.load(path)
        print(f"  [OK] Loaded {name}")
    else:
        print(f"  [MISSING] {path}")

# Các cặp cần so sánh: (model1, model2) = test xem model2 khác model1 có ý nghĩa không
# DM stat > 0: model2 tốt hơn (MSE nhỏ hơn); < 0: model2 tệ hơn
PAIRS = [
    ("tabular_only", "full",      "Tab-only vs Full"),
    ("tabular_only", "no_image",  "Tab-only vs No-Image (Tab+Text)"),
    ("tabular_only", "no_text",   "Tab-only vs No-Text (Tab+Image)"),
    ("full",         "no_image",  "Full vs No-Image"),
    ("full",         "no_text",   "Full vs No-Text"),
]

TARGETS = [
    ("y_baseline",       8, "y_funding (t+8h)"),
    ("y_heuristic",      1, "y_heuristic (t+1h)"),
    ("y_vol_adj_return", 1, "y_vol_adj_return (t+1h)"),
]

print()
print("=" * 70)
print("ABLATION DM TEST (kiểm định thống kê giữa các ablation variants)")
print("=" * 70)
print("  DM stat > 0 → model2 tốt hơn model1 | * p<0.05  ** p<0.01  *** p<0.001")
print()

for tgt_key, h, tgt_label in TARGETS:
    y_true = baseline_preds[f"{tgt_key}_y_true"]
    print(f"Target: {tgt_label}  (h={h})")
    print(f"  {'Comparison':<38} {'DM stat':>8}  {'p-value':>8}  {'Sig':>5}")
    print(f"  {'-'*38}  {'-'*8}  {'-'*8}  {'-'*5}")
    for m1, m2, label in PAIRS:
        if m1 not in ablation_preds or m2 not in ablation_preds:
            print(f"  {label:<38}   MISSING")
            continue
        p1 = ablation_preds[m1][f"{tgt_key}_MFN"]
        p2 = ablation_preds[m2][f"{tgt_key}_MFN"]
        dm, pval = diebold_mariano_test(y_true, p1, p2, h=h)
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        print(f"  {label:<38}  {dm:>8.3f}  {pval:>8.4f}  {sig:>5}")
    print()


In [ ]:
# ==========================================
# 4. DM TEST CHO EXTENDED TABULAR VARIANTS
# ==========================================
# So sánh tab_ext_only (11f) vs tabular_only (7f)  → Comparison A
# So sánh tab_ext_no_image (11f) vs full (7f)       → Comparison B
#
# LƯU Ý: Trước khi chạy, đổi tên file .npz trên Kaggle:
#   mfn_test_predictions_tabular_only.npz  (từ run --ablation tabular_only --tabular-file extended)
#   → đổi thành: mfn_test_predictions_tab_ext_only.npz
#
#   mfn_test_predictions_no_image.npz      (từ run --ablation no_image --tabular-file extended)
#   → đổi thành: mfn_test_predictions_tab_ext_no_image.npz

import os

BASE_DIR = "/kaggle/input/datasets/namkhanhng/siginificant-test"  # <<< SỬA NẾU CẦN

baseline_preds = np.load(f"{BASE_DIR}/baseline_predictions.npz")

ext_files = {
    "tabular_only"      : f"{BASE_DIR}/mfn_test_predictions_tabular_only.npz",       # base 7f
    "full"              : f"{BASE_DIR}/mfn_test_predictions_full.npz",                # base 7f
    "tab_ext_only"      : f"{BASE_DIR}/mfn_test_predictions_tab_ext_only.npz",        # extended 11f
    "tab_ext_no_image"  : f"{BASE_DIR}/mfn_test_predictions_tab_ext_no_image.npz",    # extended 11f
}

ext_preds = {}
for name, path in ext_files.items():
    if os.path.exists(path):
        ext_preds[name] = np.load(path)
        print(f"  [OK] Loaded {name}")
    else:
        print(f"  [MISSING] {path}")

# Comparison A: standalone predictive power of MA/RSI/MACD
# Comparison B: TIs as substitute for ViT
EXT_PAIRS = [
    ("tabular_only",  "tab_ext_only",      "Comp A: tabular_only (7f) vs tab_ext_only (11f)"),
    ("full",          "tab_ext_no_image",  "Comp B: full (7f) vs tab_ext_no_image (11f)"),
]

TARGETS = [
    ("y_baseline",       8, "y_funding (t+8h)"),
    ("y_heuristic",      1, "y_heuristic (t+1h)"),
    ("y_vol_adj_return", 1, "y_vol_adj_return (t+1h)"),
]

print()
print("=" * 75)
print("EXTENDED TABULAR DM TEST")
print("=" * 75)
print("  DM stat > 0 → model2 (extended) tốt hơn model1 | * p<0.05  ** p<0.01  *** p<0.001")
print()

for tgt_key, h, tgt_label in TARGETS:
    y_true = baseline_preds[f"{tgt_key}_y_true"]
    print(f"Target: {tgt_label}  (h={h})")
    print(f"  {'Comparison':<52} {'DM stat':>8}  {'p-value':>8}  {'Sig':>5}")
    print(f"  {'-'*52}  {'-'*8}  {'-'*8}  {'-'*5}")
    for m1, m2, label in EXT_PAIRS:
        if m1 not in ext_preds or m2 not in ext_preds:
            print(f"  {label:<52}   MISSING")
            continue
        p1 = ext_preds[m1][f"{tgt_key}_MFN"]
        p2 = ext_preds[m2][f"{tgt_key}_MFN"]
        dm, pval = diebold_mariano_test(y_true, p1, p2, h=h)
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        print(f"  {label:<52}  {dm:>8.3f}  {pval:>8.4f}  {sig:>5}")
    print()


In [ ]:
# ==========================================
# 5. DM TEST CHO NO-FUNDING-RATE VARIANTS
# ==========================================
# Kiểm định xem việc xoá funding_rate có làm thay đổi hiệu suất có ý nghĩa không
#
# Comp C: tabular_only (7f) vs tabular_only_no_funding (6f)
#         → kỳ vọng DM << 0 (xoá funding_rate làm model sụp đổ)
# Comp D: full (7f) vs full_no_funding (6f)
#         → kỳ vọng DM > 0 trên y_funding (full_no_funding cải thiện nhẹ)
#
# LƯU Ý: Sau khi train xong trên Kaggle, đổi tên file .npz:
#   mfn_test_predictions_tabular_only.npz  (từ run tabular_only + no_funding features)
#   → đổi thành: mfn_test_predictions_tabular_only_no_funding.npz
#
#   mfn_test_predictions_full.npz          (từ run full + no_funding features)
#   → đổi thành: mfn_test_predictions_full_no_funding.npz

import os

BASE_DIR = "/kaggle/input/datasets/namkhanhng/siginificant-test"  # <<< SỬA NẾU CẦN

baseline_preds = np.load(f"{BASE_DIR}/baseline_predictions.npz")

nf_files = {
    "tabular_only"             : f"{BASE_DIR}/mfn_test_predictions_tabular_only.npz",
    "full"                     : f"{BASE_DIR}/mfn_test_predictions_full.npz",
    "tabular_only_no_funding"  : f"{BASE_DIR}/mfn_test_predictions_tabular_only_no_funding.npz",
    "full_no_funding"          : f"{BASE_DIR}/mfn_test_predictions_full_no_funding.npz",
}

nf_preds = {}
for name, path in nf_files.items():
    if os.path.exists(path):
        nf_preds[name] = np.load(path)
        print(f"  [OK] Loaded {name}")
    else:
        print(f"  [MISSING] {path}")

# DM stat > 0 → model2 (no_funding) tốt hơn model1
NF_PAIRS = [
    ("tabular_only", "tabular_only_no_funding", "Comp C: tabular_only (7f) vs tabular_only_no_funding (6f)"),
    ("full",         "full_no_funding",          "Comp D: full (7f) vs full_no_funding (6f)"),
]

TARGETS = [
    ("y_baseline",       8, "y_funding (t+8h)"),
    ("y_heuristic",      1, "y_heuristic (t+1h)"),
    ("y_vol_adj_return", 1, "y_vol_adj_return (t+1h)"),
]

print()
print("=" * 75)
print("NO-FUNDING-RATE DM TEST")
print("=" * 75)
print("  DM stat > 0 → no_funding variant tốt hơn | * p<0.05  ** p<0.01  *** p<0.001")
print()

for tgt_key, h, tgt_label in TARGETS:
    y_true = baseline_preds[f"{tgt_key}_y_true"]
    print(f"Target: {tgt_label}  (h={h})")
    print(f"  {'Comparison':<55} {'DM stat':>8}  {'p-value':>8}  {'Sig':>5}")
    print(f"  {'-'*55}  {'-'*8}  {'-'*8}  {'-'*5}")
    for m1, m2, label in NF_PAIRS:
        if m1 not in nf_preds or m2 not in nf_preds:
            print(f"  {label:<55}   MISSING")
            continue
        p1 = nf_preds[m1][f"{tgt_key}_MFN"]
        p2 = nf_preds[m2][f"{tgt_key}_MFN"]
        dm, pval = diebold_mariano_test(y_true, p1, p2, h=h)
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else "n.s."
        print(f"  {label:<55}  {dm:>8.3f}  {pval:>8.4f}  {sig:>5}")
    print()


In [ ]:
# ==========================================
# 6. FUNDING-RATE HORIZON ROBUSTNESS TEST (t+8h / t+16h / t+24h)
# ==========================================
# Kiểm tra xem kết luận "Persistence mạnh, model gần như không vượt được"
# có còn đúng ở horizon khác ngoài t+8h hay không — cho cả tabular_only lẫn full.
#
# CHUẨN BỊ TRƯỚC KHI CHẠY:
#   1. Chạy run_baselines(features_dir=..., funding_horizon=16, out_dir="./h16")
#      → lấy file "baseline_predictions.npz" trong ./h16, đổi tên thành:
#        baseline_predictions_h16.npz
#      Lặp lại tương tự cho funding_horizon=24 → baseline_predictions_h24.npz
#   2. Bạn cần mfn_test_predictions_tabular_only_h16.npz, _h24.npz VÀ
#      mfn_test_predictions_full_h16.npz, _h24.npz (train.py tự đặt tên theo
#      horizon, không cần đổi tên) — tức là 2 ablation (tabular_only, full) x
#      2 horizon mới (16, 24) = 4 lần chạy train.py.
#   3. Upload tất cả các file trên vào cùng dataset Kaggle với baseline_predictions.npz (h8).

import os

BASE_DIR = "/kaggle/input/datasets/namkhanhng/siginificant-test"  # <<< SỬA NẾU CẦN
ABLATIONS = ["tabular_only", "full"]
HORIZONS = [8, 16, 24]

def r2_oos_vs_benchmark(preds, y_true, benchmark_preds):
    ss_res = np.sum((preds - y_true) ** 2)
    ss_tot = np.sum((y_true - benchmark_preds) ** 2)
    return 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

print("=" * 75)
print("FUNDING-RATE HORIZON ROBUSTNESS: Persistence vs MFN (tabular_only & full)")
print("=" * 75)

for h in HORIZONS:
    baseline_path = f"{BASE_DIR}/baseline_predictions_h{h}.npz" if h != 8 else f"{BASE_DIR}/baseline_predictions.npz"
    if not os.path.exists(baseline_path):
        print(f"\n[h={h}h] MISSING: {baseline_path}")
        continue
    baseline_preds = np.load(baseline_path)
    y_true = baseline_preds["y_baseline_y_true"]
    y_persist = baseline_preds["y_baseline_Persistence"]
    y_histmean = baseline_preds["y_baseline_HistoricalMean"]
    persistence_r2_oos = r2_oos_vs_benchmark(y_persist, y_true, y_histmean)

    print(f"\nHorizon = {h}h  (n={len(y_true)})")
    print(f"  Persistence R²_OOS (vs Historical Mean) : {persistence_r2_oos:.4f}")
    print(f"  {'Ablation':<15} {'R²_OOS vs Persistence':>22}  {'DM stat':>8}  {'p-value':>8}  {'Sig':>5}")
    print(f"  {'-'*15} {'-'*22}  {'-'*8}  {'-'*8}  {'-'*5}")

    for ablation in ABLATIONS:
        mfn_path = f"{BASE_DIR}/mfn_test_predictions_{ablation}_h{h}.npz"
        if not os.path.exists(mfn_path):
            print(f"  {ablation:<15}   MISSING ({mfn_path})")
            continue
        mfn_preds = np.load(mfn_path)
        y_mfn = mfn_preds["y_baseline_MFN"]

        if "y_baseline_y_true" in mfn_preds:
            mfn_y_true = mfn_preds["y_baseline_y_true"]
            if len(mfn_y_true) == len(y_true) and np.max(np.abs(mfn_y_true - y_true)) > 1e-2:
                print(f"  ⚠ [{ablation}, h={h}h] WARNING: baseline y_true != MFN y_true "
                      f"(max diff={np.max(np.abs(mfn_y_true - y_true)):.4f}). "
                      f"Check funding_horizon/features_dir match between the two runs!")

        mfn_r2_oos = r2_oos_vs_benchmark(y_mfn, y_true, y_persist)
        dm_stat, dm_pval = diebold_mariano_test(y_true, y_persist, y_mfn, h=h)
        sig = "***" if dm_pval < 0.001 else "**" if dm_pval < 0.01 else "*" if dm_pval < 0.05 else "n.s."
        print(f"  {ablation:<15} {mfn_r2_oos:>22.4f}  {dm_stat:>8.4f}  {dm_pval:>8.4f}  {sig:>5}")
